# 文本分类实例

## S1 导入包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## S2 加载数据

In [2]:
import pandas as pd

data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [3]:
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


## S3 创建Dataset

In [4]:
from torch.utils.data import Dataset
from pathlib import Path


class MyDataset(Dataset):
    def __init__(self, data_path: Path) -> None:
        super().__init__()
        self.data: pd.DataFrame = pd.read_csv(data_path)
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]

    def __len__(self):
        return len(self.data)

In [5]:
dataset = MyDataset(Path("./ChnSentiCorp_htl_all.csv"))
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', np.int64(1))
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', np.int64(1))
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', np.int64(1))
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', np.int64(1))
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', np.int64(1))


## S4 划分数据集

In [6]:
from torch.utils.data import random_split

trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [7]:
for i in range(10):
    print(trainset[i])

('在庐山这样的名胜旅游区，这样的酒店已经不错。地点很好，特别是自驾游的旅客，上下山都很方便。只是好象节能有点过了头，走廊没开灯。携程说有扑克送也没有送给我们。补充点评2007年9月28日：对了，卫生条件一般。有电脑上网，这一点对我们这些年轻一点的旅游者来说是很好的事情，毕竟下一站的定房，路线查询都需要上网才能解决。', np.int64(1))
('我们一家三口在岛上住了一天,总的感觉还不错.沙滩海水无得弹,很多珊湖石.在携程订观海木楼,山下几间房是看不到海的,一定要山上的才能观到海,有点遗憾.在岛上吃得还能接受,就是什么都要现金,另外晚上有很多小动物.', np.int64(1))
('请大家千万别住别墅，别墅的空调特差，很难制冷，而且经常半夜断电。酒店前台的服务意识很差，态度极为不好，明明有制冷好的主楼房间，就是不给携程的客户住。如果前台问你们别墅的房间可以不可以，一定不能同意。补充点评2008年7月15日：另外，早餐不太好，我们一般7点半去吃就没剩什么东西了，和团餐标准一样简单的不能再简单了。', np.int64(0))
('酒店位置还不错，环境也可以，房间小了点。', np.int64(1))
('服务态度尚可;房间硬件较老;酒店外部环境不错,空气清新,安静.坐落在山坡上.', np.int64(1))
('都是如家快捷还真是千差万别。这家店一进门便是一股难闻的怪味，房间里面充斥着下水沟的味道。早餐更不敢恭维。只是为了办事方便。', np.int64(0))
('服务很不错，服务员也很有礼貌。房间宽大，设施非常全，房间拖鞋不错，比其它酒店的好多了。酒店位置也很好，购物，娱乐，都很方便。总体很好。', np.int64(1))
('在衢州能住到这样的酒店算是很不错了，房间液晶电视，窗口看出去是条小河，还送了水果，算是很不错的啦，就是周围环境一般，没什么可以逛的...', np.int64(1))
('1。我住的是靠马路的标准间。房间内设施简陋，并且的房间玻璃窗户外还有一层幕墙玻璃，而且不能打开，导致房间不能自然通风，采光不好。2。吃了三顿早餐，品种少得可怜。3。二楼餐厅被租出去了，价格特贵。原来订餐时告之房客可以打九折（房间服务指南里也明明白白写的，等到结帐时却告之海鲜和酒水是不能打折的，而且没有发票，找来经理才好不容易在第二天拿到发票。综合来说与四星级差

## S5 创建DataLoader

In [ ]:
import torch

tokenizer = AutoTokenizer.from_pretrained("rbt3")

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = list(texts)
    labels = list(labels)
    inputs = tokenizer(
        texts,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )  # ty:ignore[call-non-callable]
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(trainset, batch_size=16, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(validset, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [10]:
next(iter(train_loader))

{'input_ids': tensor([[ 101, 6821, 2157,  ...,    0,    0,    0],
        [ 101,  817, 3419,  ..., 2137, 2918,  102],
        [ 101, 7471, 2270,  ...,    0,    0,    0],
        ...,
        [ 101, 1157, 1343,  ..., 6983, 2421,  102],
        [ 101, 2791, 7313,  ...,    0,    0,    0],
        [ 101, 1068,  754,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1])}

## S6 创建模型以及优化器

In [11]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("rbt3")

optimizer = Adam(model.parameters(), lr=2e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: rbt3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training o

## S7 训练与验证

In [12]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            pred = torch.argmax(outputs.logits, dim=-1)
            acc_num += (pred.long() == batch["labels"].long()).float().sum().item()
    return acc_num / len(validset)


def train(epochs: int = 3, log_step: int = 100):
    global_step = 0
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            if torch.cuda.is_available():
                batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            outputs.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep:{epoch}, global_step:{global_step}, loss:{outputs.loss}")
            global_step += 1
        acc = evaluate()
        print(f"ep:{epoch}, acc:{acc}")

## S8 模型训练

In [13]:
train()

ep:0, global_step:0, loss:0.8097412586212158
ep:0, global_step:100, loss:0.4860695004463196
ep:0, global_step:200, loss:0.33215439319610596
ep:0, global_step:300, loss:0.39477407932281494
ep:0, global_step:400, loss:0.17793019115924835
ep:0, acc:0.8891752577319587
ep:1, global_step:500, loss:0.07270676642656326
ep:1, global_step:600, loss:0.5278140306472778
ep:1, global_step:700, loss:0.032385773956775665
ep:1, global_step:800, loss:0.07791659235954285
ep:1, acc:0.8904639175257731
ep:2, global_step:900, loss:0.25222018361091614
ep:2, global_step:1000, loss:0.0495653934776783
ep:2, global_step:1100, loss:0.4400021433830261
ep:2, global_step:1200, loss:0.1142488420009613
ep:2, global_step:1300, loss:0.07749748229980469
ep:2, acc:0.8814432989690721


## S9 预测

In [ ]:
sen = "我还想来这个酒店呢！"
model.eval()
id2label = {0: "差评", 1: "好评"}
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt").to(device)  # ty:ignore[call-non-callable]
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"input: {sen} \n pred: {id2label.get(int(pred.item()))}")


input: 我还想来这个酒店呢！ 
 pred: 好评


In [22]:
from transformers import pipeline

model.config.id2label = id2label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [23]:
pipe(sen)

[{'label': '好评', 'score': 0.6412907242774963}]